# Baseline Notebook — Customer Churn

Purpose: establish non-ML and simple-ML baselines on `customer-churn-training.csv` **before** any decision to invest in a production model. See the problem-framing memo and data card for the reasoning behind this order of operations.

**n = 12 rows.** Every result below should be read as illustrative of method, not as a reliable performance estimate.

In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv('customer-churn-training.csv')
df


   customer_id  tenure_months  support_tickets  monthly_spend_inr  \
0         C001              2                4                499   
1         C002             18                1               1299   
2         C003              6                3                799   
3         C004             30                0               1499   
4         C005             11                2                999   
5         C006              1                5                499   
6         C007             24                1                999   
7         C008              8                4                799   
8         C009             15                2               1299   
9         C010              4                3                499   
10        C011             20                0                999   
11        C012              7                2                799   

    last_login_days plan_type  churned  
0                21     Basic        1  
1                 2 

## 1. Data quality checks

In [ ]:
print('Shape:', df.shape)
print('\nMissing values per column:')
print(df.isna().sum())
print('\nDuplicate rows:', df.duplicated().sum())
print('\nClass balance:')
print(df['churned'].value_counts())
print('\nChurn rate by plan_type:')
print(df.groupby('plan_type')['churned'].mean())


Shape: (12, 7)

Missing values per column:
customer_id          0
tenure_months        0
support_tickets      0
monthly_spend_inr    0
last_login_days      0
plan_type            0
churned              0
dtype: int64

Duplicate rows: 0

Class balance:
churned
0    7
1    5
Name: count, dtype: int64

Churn rate by plan_type:
plan_type
Basic       1.0
Pro         0.0
Standard    0.0
Name: churned, dtype: float64


**Finding:** no missing values, no duplicates. But `plan_type` perfectly separates the target (every Basic customer churned, no Standard/Pro customer did). That's the first sign this file is a small illustrative sample rather than a representative production export — flagged in the data card.

## 2. Non-ML baselines

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, confusion_matrix

y = df['churned'].values
n = len(y)

# Baseline 1: majority class
majority_pred = np.zeros(n, dtype=int)
print('=== Baseline 1: Majority class (always predict not-churned) ===')
print('Accuracy: ', accuracy_score(y, majority_pred))
print('Precision:', precision_score(y, majority_pred, zero_division=0))
print('Recall:   ', recall_score(y, majority_pred, zero_division=0))


=== Baseline 1: Majority class (always predict not-churned) ===
Accuracy:  0.5833333333333334
Precision: 0.0
Recall:    0.0


In [ ]:
# Baseline 2: simple business rule on recency
threshold = 10
rule_pred = (df['last_login_days'] > threshold).astype(int).values
print(f'=== Baseline 2: Rule (last_login_days > {threshold} => churn) ===')
print('Accuracy: ', accuracy_score(y, rule_pred))
print('Precision:', precision_score(y, rule_pred, zero_division=0))
print('Recall:   ', recall_score(y, rule_pred, zero_division=0))
print('Confusion matrix:\n', confusion_matrix(y, rule_pred))


=== Baseline 2: Rule (last_login_days > 10 => churn) ===
Accuracy:  1.0
Precision: 1.0
Recall:    1.0
Confusion matrix:
 [[7 0]
 [0 5]]


In [ ]:
# Baseline 3: simple business rule on plan tier
plan_pred = (df['plan_type'] == 'Basic').astype(int).values
print("=== Baseline 3: Rule (plan_type == 'Basic' => churn) ===")
print('Accuracy: ', accuracy_score(y, plan_pred))
print('Precision:', precision_score(y, plan_pred, zero_division=0))
print('Recall:   ', recall_score(y, plan_pred, zero_division=0))


=== Baseline 3: Rule (plan_type == 'Basic' => churn) ===
Accuracy:  1.0
Precision: 1.0
Recall:    1.0


**Finding:** two single-feature business rules already reach 100% accuracy on this file. Any ML model must be judged against these, not against the majority-class baseline alone.

## 3. Simple ML baseline (logistic regression, leave-one-out CV)
With only 12 rows, a train/test split would leave single-digit samples per class, so leave-one-out cross-validation is used instead. This still should not be read as a reliable performance estimate — it's included to show the *method*, to be repeated properly once a real dataset is available.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import LeaveOneOut

X = pd.get_dummies(
    df[['tenure_months','support_tickets','monthly_spend_inr','last_login_days','plan_type']],
    columns=['plan_type'], drop_first=True
)

loo = LeaveOneOut()
preds = np.zeros(n, dtype=int)
for train_idx, test_idx in loo.split(X):
    model = LogisticRegression(max_iter=1000)
    model.fit(X.iloc[train_idx], y[train_idx])
    preds[test_idx] = model.predict(X.iloc[test_idx])

print('=== Logistic Regression (Leave-One-Out CV) ===')
print('Accuracy: ', accuracy_score(y, preds))
print('Precision:', precision_score(y, preds, zero_division=0))
print('Recall:   ', recall_score(y, preds, zero_division=0))
print('Confusion matrix:\n', confusion_matrix(y, preds))


=== Logistic Regression (Leave-One-Out CV) ===
Accuracy:  0.9166666666666666
Precision: 0.8333333333333334
Recall:    1.0
Confusion matrix:
 [[6 1]
 [0 5]]


## 4. Comparison table

| Approach | Accuracy | Precision | Recall |
|---|---|---|---|
| Majority class | 58.3% | 0.00 | 0.00 |
| Rule: last_login_days > 10 | 100% | 1.00 | 1.00 |
| Rule: plan_type == Basic | 100% | 1.00 | 1.00 |
| Logistic regression (LOOCV) | 91.7% | 0.83 | 1.00 |

## 5. Conclusion
On this file, the ML model does **not** beat the simple business rules — it's worse. Combined with the perfect class separation and n=12 sample size (see the data card), the recommendation is to **not** proceed to a production ML model on this data. Ship and monitor the simple rule instead, and revisit ML only once a larger, properly time-stamped dataset is available (see the problem-framing memo, section 6, for full reasoning).